[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/reciprocal/reciprocal.ipynb)

# Lifts and Reciprocal Diagrams

This example computes the reciprocal diagram of a planar truss from a lift of its mesh. Give every vertex of a planar triangle mesh a height and the mesh lifts to a polyhedral terrain. Read the terrain as an Airy stress function and its creases are the bars of a planar truss: how sharply the terrain bends across an edge is the force in that bar, tension where it bends convex and compression where it bends concave, and every interior vertex is in equilibrium by construction. The slopes of the terrain's faces are the vertices of a second mesh, the reciprocal diagram, with an edge across every bar, perpendicular to it and as long as the bar's force. Lifted onto the paraboloid, the reciprocal diagram is the Voronoi diagram and the forces are the cotangent weights.

The vertices are points, antivectors, intrinsic like all geometry, and a face of the terrain is a plane, a vector: their pairing, `plane & (point + UP)`, with the point lifted to unit height, is the height there. Through three corners the plane is their heights times the reciprocal frame of the lifted corners, the corners met in pairs over all three met, `(b & c) / (a & b & c)`, linear in the heights. The reciprocal diagram's vertices are those planes with their height at the origin wedged away, `(plane ^ mv.z) * mv.z`, and a bar's stress is the jump in slope across it, paired with the corner across the bar.

In matrix notation the bar forces read as the entries of an equilibrium stress, a symmetric matrix $\Omega$ with zero row sums whose product with the matrix of vertex positions vanishes at every interior vertex, $\Omega P = 0$; the Maxwell–Cremona correspondence pairs every such stress with a lift.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import VGA3D as ga
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.reciprocal import render

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, mv = mesh_module.Mesh, mesh_module.mv
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Point = mesh_module.Point
# The point one unit above the origin: added to a point of the plane, it lifts it to unit height.
UP = mv.z.dual()

# The grid: its vertices per side across the square from -1 to 1, how far its interior vertices are nudged,
# and the seed of the nudges.
n_side = 5
jitter = 0.08
grid_seed = 11
across = np.linspace(-1, 1, n_side)
coords = np.stack(np.meshgrid(across, across, indexing="ij"), axis=-1).reshape(-1, 2)   # [vertices, 2]
inside = (np.abs(coords) < 1).all(axis=-1)
coords[inside] += jitter * np.random.default_rng(grid_seed).normal(size=(inside.sum(), 2))
# Points of the plane, at height zero, and the mesh through them.
points = mv.antivector(np.concatenate([coords, np.zeros((len(coords), 1))], axis=-1)).field()   # Point[vertices]
mesh = Mesh._triangulate(points)

## 1. Slopes, stresses and the reciprocal diagram

A face of the terrain is a plane, and a plane over the plane of the mesh is one vector, its slope and its height at the origin, whose pairing with a point lifted to unit height, `plane & (point + UP)`, is the height there. Two lifted corners meet in a vector that pairs to nothing with either; over the meet of all three it pairs to one with the third, so the plane through the three corners at their heights is their heights times those three vectors, the reciprocal frame of the corners: linear in the heights. Across an interior edge the slope jumps, and the jump pairs to nothing with the edge, since both planes agree along it; over the edge's length it is the bar's stress. The faces' slopes are the vertices of the reciprocal diagram, and the jumps its edges. On the paraboloid, at half the squared distance from the centre, each face's slope, read as a point by the metric, `slope.dual()`, is its circumcentre and each stress the edge's cotangent weight, half the cotangents of the two angles facing it; the bars across cocircular corners carry almost nothing.

In [ ]:
def slopes(mesh: Mesh, points: Point, heights: np.ndarray) -> Vector:
    """The plane of each face of the terrain, through its corners at their heights, as the vector whose
    pairing with a point lifted to unit height, `point + UP`, is the height there: the slope across the
    plane and the height at the origin. The corners' heights times the reciprocal frame of the corners
    lifted to unit height, the three vectors pairing to one with their own corner and to nothing with the
    other two: linear in the heights, the heights on the last axis."""
    lifted = points.batch() + UP                                               # [vertices] Point
    a, b, c = (lifted[mesh.faces[:, corner]] for corner in range(3))          # [faces] Point each
    # Two corners meet in a vector that pairs to nothing with either; over the meet of all three, it pairs
    # to one with the third.
    volume = a & b & c                                                         # [faces] Scalar
    frame = stack([(b & c) / volume, (c & a) / volume, (a & b) / volume], axis=-1)   # [faces, 3] Vector
    return (frame * heights[..., mesh.faces]).sum(axis=-1)                     # [..., faces] Vector



def beside(mesh: Mesh) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The interior edges, and the faces on their left and on their right, seen along the edge from its
    lower vertex to its higher."""
    edge = mesh.face_edges.reshape(-1)                                         # [faces * 3]
    face = np.repeat(np.arange(len(mesh.faces)), 3)                            # [faces * 3]
    along = mesh.face_edge_orientation.reshape(-1) > 0                        # [faces * 3]
    left, right = np.full(len(mesh.edges), -1), np.full(len(mesh.edges), -1)
    left[edge[along]], right[edge[~along]] = face[along], face[~along]
    inner = np.flatnonzero((left >= 0) & (right >= 0))
    return inner, left[inner], right[inner]


def stresses(mesh: Mesh, points: Point, heights: np.ndarray) -> Scalar:
    """The stress of each interior edge: the jump in slope between the faces on either side, measured
    towards the corner across the edge, over the edge's length, positive in tension, where the terrain
    bends convex. Which face is called left does not matter: swapping them turns both the jump and the
    side the corner lies on."""
    inner, left, right = beside(mesh)                                          # [inner] each
    planes = slopes(mesh, points, heights)                                     # [..., faces] Vector
    jump = planes[..., left] - planes[..., right]                              # [..., inner] Vector
    # Each bar from its tail to its head.
    bars = points.batch()[mesh.edges[inner, 1]] - points.batch()[mesh.edges[inner, 0]]   # [inner] Point
    # The corner across each edge in the face on its right, as its offset square to the bar.
    far = mesh.faces[right].sum(axis=-1) - mesh.edges[inner].sum(axis=-1)    # [inner]
    corner = points.batch()[far] - points.batch()[mesh.edges[inner, 0]]       # [inner] Point
    across = corner - bars * ((bars | corner) / (bars | bars))                # [inner] Point
    # The jump in slope paired with that offset: how much faster the terrain rises across the bar.
    return -(jump & across) / (across.norm() * bars.norm())                    # [..., inner] Scalar


def folds(mesh: Mesh, points: Point, heights: np.ndarray) -> Scalar:
    """How far the corner across each interior edge on its right lies above the plane of the face on its
    left: positive where the terrain bends convex."""
    inner, left, right = beside(mesh)                                          # [inner] each
    far = mesh.faces[right].sum(axis=-1) - mesh.edges[inner].sum(axis=-1)    # [inner]
    lifted = points.batch()[far] + UP                                          # [inner] Point
    return heights[..., far] - (slopes(mesh, points, heights)[..., left] & lifted)   # [..., inner] Scalar


paraboloid = (coords**2).sum(axis=-1) / 2                                      # [vertices]
inner, left, right = beside(mesh)
lifted_stresses = stresses(mesh, points, paraboloid)                           # [inner] Scalar
lifted_duals = (slopes(mesh, points, paraboloid) ^ mv.z) * mv.z                # [faces] Vector

In [ ]:
save_figure(render.draw(mesh, points, paraboloid, lifted_stresses, lifted_duals), "reciprocal");

## 2. Tension and compression

Now let every height breathe a little about the paraboloid, each with its own phase. Where the terrain still bends convex the bar stays in tension, blue; where a vertex rises far enough above its neighbours the terrain creases concave there, the bar goes into compression, red, and its reciprocal edge turns around, folding the reciprocal diagram over itself. Every interior vertex stays in equilibrium throughout, and every reciprocal edge stays across its bar: the heights only ever change how hard each bar pulls or pushes. The boundary vertices are where the truss is held, and the forces there are its loads.

In the notation of graphic statics, the planar truss is the form diagram and the reciprocal diagram the force diagram: each closed polygon of the force diagram is the equilibrium of the joint it is drawn across, and the polyhedral terrain over the form diagram is its Maxwell lift.

In [ ]:
# How far each height breathes about the paraboloid, the seed of the phases, the frames of one breath and
# the duration of each.
amplitude = 0.02
phase_seed = 3
n_frames = 48
duration_ms = 80

In [ ]:
phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, len(paraboloid))
times = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None]
breaths = paraboloid + amplitude * np.sin(times + phases)                      # [frames, vertices]
breath_stresses = stresses(mesh, points, breaths)                              # [frames] Scalar[inner]
breath_duals = (slopes(mesh, points, breaths) ^ mv.z) * mv.z                   # [frames, faces] Vector

In [ ]:
compressed = (breath_stresses < 0).sum(axis=-1)
print(f"bars in compression over the breath: {compressed.min()} to {compressed.max()} of {len(inner)}")
frames = ((breath, stresses(mesh, points, breath), (slopes(mesh, points, breath) ^ mv.z) * mv.z) for breath in breaths)
show_animation(render.animate(mesh, points, frames), "reciprocal_breath", duration_ms)

In [ ]:
# checks
# Every bar is in tension exactly where the terrain bends convex, and some bars go into compression.
assert np.array_equal(breath_stresses.to_array() > 0, folds(mesh, points, breaths).to_array() > 0)
assert (breath_stresses.to_array() < 0).any()
# Every reciprocal edge pairs to nothing with its bar, and every interior vertex is in equilibrium.
bars = (mesh.d0 * points).batch()[inner]                                       # [inner] Point
assert np.abs(((breath_duals[:, left] - breath_duals[:, right]) & bars).to_array()).max() < 1e-10
# The bars meeting at a vertex each pull it along themselves by their stress.
pulls = (bars * breath_stresses).field()                                       # [frames] Point[inner]
tails, heads = (mesh.reading(mesh.edges[inner, end]) for end in range(2))      # [inner, vertices] Scalar each
net = (~tails * pulls - ~heads * pulls).batch()                                # [frames, vertices] Point
interior = np.setdiff1d(np.arange(mesh.n_vertices), mesh.edges[mesh.boundary_edges])
assert net[:, interior].norm().to_array().max() < 1e-10
# On the paraboloid the stresses are the cotangent weights, half the cotangents of the angles facing each edge,
# and the reciprocal vertices the circumcentres.
edges = ((mesh.d0 * points).batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()   # [3] Point[faces]
after, before = edges[[1, 2, 0]], edges[[2, 0, 1]]                             # [3] Point[faces] each
cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar[faces]
cotangent_weights = ~mesh.by_corner(cotangents / 2) * mesh_module.as_scalar(np.ones(len(mesh.faces))).field()   # Scalar[edges]
assert np.abs((lifted_stresses - cotangent_weights.batch()[inner]).to_array()).max() < 1e-10
reach = [(lifted_duals.dual() - corner).scalar_norm_squared().to_array() for corner in points.batch()[mesh.faces.T]]
assert np.abs(reach[0] - reach[1]).max() < 1e-10 and np.abs(reach[0] - reach[2]).max() < 1e-10